# Building Agents with Anthropic's AgentSkill Structure in Vinagent

We'll thrill to announce an new exciting feature in **Vinagent** that allows everybody to build agents using the [Anthropic AgentSkill](https://platform.claude.com/docs/en/agents-and-tools/agent-skills/overview) structure. This approach enables to define tools entirely through documentation (`SKILL.md`) without needing to write complex Python tool wrappers. Therefore, Vinagent's agents can approach a wide range of skills that are widely shared in the community. Afterwards, LLM learns how to operate the skill just by reading the documentation and executes dynamically generated Python code or Bash commands in a safe, designated working directory.

## 1. What is an AgentSkill?

An AgentSkill is a directory representing a self-contained capability for an agent. It follows a specific structure popularized by Anthropic:

```text
my_skill/
├── SKILL.md       # YAML front-matter (name, description) + markdown usage guidance
└── scripts/       # (Optional) Helper scripts utilized by the skill
```

Unlike traditional tools where you register specific Python functions, an AgentSkill uses the `SKILL.md` file as its source of truth. The LLM reads the examples and instructions inside `SKILL.md` and generates the necessary Python code or Bash commands on the fly to fulfill your request!

This is an amazing list of AgentSkills repositories curated and tailored for common tasks across various domains, making them readily available for your use.

1. [anthropics/skills](https://github.com/anthropics/skills): This repository provides official examples of AgentSkills following the open standard proposed for extending AI agents. It demonstrates best practices for structuring skills, defining instructions, and integrating tools within agent workflows. Developers can use it as a reference when designing their own reusable skills for Claude-based or compatible agents.

2. [VoltAgent/awesome-agent-skills](https://github.com/VoltAgent/awesome-agent-skills): A large curated directory of AgentSkills repositories and individual skills contributed by the community. It aggregates skills from companies and developers across the ecosystem, making it one of the best starting points for discovering reusable capabilities for AI agents. The repository organizes skills by category and supports many agent frameworks such as Claude Code, Codex, and Cursor.

3. [skillcreatorai/Ai-Agent-Skills](https://github.com/skillcreatorai/Ai-Agent-Skills): This repository provides a structured collection of reusable AgentSkills designed to extend the capabilities of AI coding agents. Each skill follows a standardized folder structure with documentation, scripts, and references that agents can use to perform specific tasks. It also supports CLI-based installation so developers can easily add skills like frontend design, code review, or database design to their agents.

4. [tech-leads-club/agent-skills](https://github.com/tech-leads-club/agent-skills): A curated set of engineering-oriented skills intended for professional software development workflows. The repository focuses on practical capabilities such as cloud architecture guidance, accessibility checks, security audits, and engineering best practices. These skills help AI agents assist developers with production-quality code and architecture decisions.

5. [ynulihao/AgentSkillOS](https://github.com/ynulihao/AgentSkillOS): AgentSkillOS explores how skills can be organized, retrieved, and executed within large-scale multi-agent systems. The project introduces concepts like capability trees and DAG-based orchestration to manage complex skill workflows. It is particularly relevant for researchers studying scalable AI agent ecosystems and automated task composition.

## 2. Setting up the Environment

First, let's set up our language model. We'll use OpenAI's `gpt-4o-mini` in this example. Make sure your `.env` file contains your `OPENAI_API_KEY`.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI

# Load environment variables
load_dotenv(find_dotenv('.env'))

# Initialize the LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
)
print("LLM initialized.")

## 3. Loading the Agent with AgentSkill

To use an AgentSkill, we simply point Vinagent's `Agent.load_agent` method to the directory containing our `SKILL.md`.

In this example, we have an existing skill at [vinagent/agentskills/skills/skills/xlsx](https://github.com/datascienceworld-kan/vinagent/tree/main/agentskills/skills/skills/xlsx). This skill contains instructions on how to read, write, and manipulate Excel files using `pandas` and `openpyxl`.

In [ ]:
from vinagent.agent import Agent

# Instantiate the agent using the AgentSkill directory path
agent = Agent.load_agent(
    agent_path="agentskills/skills/skills/xlsx", 
    llm=llm
)
print(f"Agent loaded successfully with name: {agent.name}")

## 4. Invoking the Agent

Now for the fun part! We can ask the agent to perform tasks related to Excel files. 

Notice how we just provide natural language. Behind the scenes, the LLM will consult the `SKILL.md` documentation, realize it needs to write an Excel file with specific headers, construct the exact Python code to do so, and execute it!

In [ ]:
# Ask the agent to create a new Excel file with specific columns
query="""Let's create a new excel file /Users/phamdinhkhanh/Desktop/output.xlsx content
    | Student_ID | Student_Name | Math | Physics | Chemistry | English | Average_Score |
| ---------- | ------------ | ---- | ------- | --------- | ------- | ------------- |
| S001       | Nguyen An    | 8.5  | 7.8     | 8.0       | 7.5     | 7.95          |
| S002       | Tran Binh    | 6.5  | 7.0     | 6.8       | 7.2     | 6.88          |
| S003       | Le Chi       | 9.0  | 8.7     | 8.5       | 8.8     | 8.75          |
| S004       | Pham Dung    | 7.2  | 6.9     | 7.5       | 7.0     | 7.15          |
| S005       | Hoang Giang  | 8.8  | 9.1     | 8.9       | 8.5     | 8.83          |
| S006       | Vu Hai       | 5.9  | 6.3     | 6.0       | 6.5     | 6.18          |
| S007       | Doan Khanh   | 7.8  | 8.0     | 7.9       | 7.6     | 7.83          |
| S008       | Bui Lan      | 9.2  | 9.0     | 9.1       | 9.3     | 9.15          |
| S009       | Dang Minh    | 6.8  | 7.2     | 6.9       | 7.1     | 7.00          |
| S010       | Phan Ngoc    | 8.1  | 7.9     | 8.3       | 8.0     | 8.08          |
"""

response = agent.invoke(query)
print(response)

## How it Works Under the Hood 🛠️

When we run the `invoke` command, Vinagent's `AgentSkillTool` steps in:
1. **Prompting:** The agent is shown the full content of `SKILL.md` as "Skill Guidance".
2. **Code Generation:** The LLM decides what Python code or Shell command to run based strictly on the examples provided in the `SKILL.md`.
3. **Execution Mode Detection:** The tool heuristic detects if the generated command is a Python script (e.g., contains `import pandas`) or a Bash command (e.g., `python scripts/unpack.py`).
4. **Execution:** 
   - **Python:** It saves the code to a temporary `.py` file and runs it via Python so imports and multi-line logic work naturally.
   - **Bash:** It runs the command directly via `subprocess`.
5. **Feedback Loop:** The terminal output (stdout/stderr) is captured and sent back to the LLM so it can analyze the result and answer your query.

This design is incredibly powerful because it **decouples tool logic from your codebase**. You can add brand new capabilities to your agent simply by writing a markdown file with code examples!